# Q7. 手写 PPO Clip Loss（单步）

用 PyTorch 实现单步 PPO clip loss，不使用任何 RL 库。

```python
log_prob_old = torch.tensor(-1.2)                        # log π_old
log_prob_new = torch.tensor(-0.5, requires_grad=True)    # log π_new
advantage    = torch.tensor(2.0)
eps          = 0.2
```

1. 计算 `ratio = exp(log_prob_new - log_prob_old)`
2. 计算 `unclipped = ratio * advantage`
3. 计算 `clipped = torch.clamp(ratio, 1-eps, 1+eps) * advantage`
4. `loss = -torch.min(unclipped, clipped)`
5. `loss.backward()`，打印 `log_prob_new.grad`

**观察**：grad 是正、负还是零？这意味着训练会让 $\pi_{\text{new}}$ 朝哪个方向走？

**验收标准**：代码运行正确，能解释 grad 符号与 policy 更新方向的关系。

## 手算预判

先算出每一步的数值，预判 grad 的符号。

**Step 1：ratio**

$$r = e^{\log\pi_{\text{new}} - \log\pi_{\text{old}}} = e^{-0.5 - (-1.2)} = e^{0.7} \approx 2.014$$

**Step 2：unclipped**

$$\text{unclipped} = 2.014 \times 2.0 \approx 4.028$$

**Step 3：clipped**

$r = 2.014$ 超出上限 $1 + \varepsilon = 1.2$，被截断：

$$\text{clipped} = \text{clamp}(2.014,\ 0.8,\ 1.2) \times 2.0 = 1.2 \times 2.0 = 2.4$$

**Step 4：loss**

$$\text{loss} = -\min(4.028,\ 2.4) = -2.4$$

**Step 5：grad 预判**

`min` 选到了 `clipped`，而 `clipped` 在 $r > 1.2$ 时是常数（`torch.clamp` 被截断的部分梯度为 0）。

所以 $\frac{\partial \text{loss}}{\partial \log\pi_{\text{new}}} = 0$。

这正是 clipping 的核心作用：**ratio 超出范围时，梯度自动归零，停止在这个方向上继续更新。**

In [ ]:
import torch

def ppo_clip_loss(log_prob_old_val, log_prob_new_val, advantage_val, eps=0.2):
    log_prob_new = torch.tensor(log_prob_new_val, requires_grad=True)
    log_prob_old = torch.tensor(log_prob_old_val)
    advantage    = torch.tensor(advantage_val)

    ratio     = torch.exp(log_prob_new - log_prob_old)
    unclipped = ratio * advantage
    clipped   = torch.clamp(ratio, 1 - eps, 1 + eps) * advantage
    loss      = -torch.min(unclipped, clipped)
    loss.backward()

    print(f"  log_prob_new={log_prob_new_val}, ratio={ratio.item():.3f}", end="")
    print(f"  | unclipped={unclipped.item():.3f}, clipped={clipped.item():.3f}", end="")
    print(f"  | min picks={'clipped' if clipped.item() <= unclipped.item() else 'unclipped'}", end="")
    print(f"  | grad={log_prob_new.grad.item():.4f}")
    return log_prob_new.grad.item()

print("Case 1: ratio > 1+eps → clipping active")
ppo_clip_loss(-1.2, -0.5, 2.0)   # ratio ≈ 2.014, outside [0.8, 1.2]

print("\nCase 2: ratio inside [0.8, 1.2] → clipping not active")
ppo_clip_loss(-1.2, -1.1, 2.0)   # ratio ≈ 1.105, inside [0.8, 1.2]

print("\nCase 3: ratio < 1-eps → clipping active on the lower bound")
ppo_clip_loss(-1.2, -2.5, 2.0)   # ratio ≈ 0.301, outside [0.8, 1.2]

## 解释：grad 符号与更新方向的关系

| grad | 含义 | π_new 的变化方向 |
|------|------|------------------|
| 负 | loss 随 $\log\pi_{\text{new}}$ 增大而减小 | 梯度下降会增大 $\log\pi_{\text{new}}$，即**提高** $\pi_{\text{new}}$ |
| 正 | loss 随 $\log\pi_{\text{new}}$ 增大而增大 | 梯度下降会减小 $\log\pi_{\text{new}}$，即**降低** $\pi_{\text{new}}$ |
| **零** | **clipping 生效** | min 选 clipped（常数），梯度为 0， **不更新，policy 停在原地** |

本题 `grad = 0`，原因是 $r = 2.014 > 1.2$，新策略已经偏离旧策略太远。
PPO 通过让梯度归零，强制「这一步就走到这里，不许再往前」。

如果把 `log_prob_new` 改小一点（让 ratio 落在 $[0.8, 1.2]$ 之内），clipping 不生效，grad 会是负数（提高概率），训练照常进行。